In [2]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

# Load the cleaned data and separate features from the target
df = pd.read_csv("used_cars_cleaned_07.csv")

X = df.drop(columns=["price_gbp", "condition"], errors="ignore")
y = df["price_gbp"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

# One-hot encode categorical columns, including version.
# Decision trees do not require numeric scaling.
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(
        max_depth=15,
        min_samples_leaf=10,
        random_state=42
    ))
])

tree_model.fit(X_train, y_train)

results = []

for split, X_part, y_part in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val)
]:
    predictions = tree_model.predict(X_part)

    results.append({
        "Dataset": split,
        "MAE (£)": mean_absolute_error(y_part, predictions),
        "RMSE (£)": np.sqrt(mean_squared_error(y_part, predictions)),
        "R²": r2_score(y_part, predictions)
    })

display(pd.DataFrame(results).round(3))

,Dataset,MAE (£),RMSE (£),R²
0,Training,2688.961,20153.510,0.635
1,Validation,3000.683,40760.608,0.271


In [3]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
)

band_names = ["Under £10,000", "£10,000–£24,999", "£25,000+"]
band_cutoffs = [10_000, 25_000]
results = []

for split, X_part, y_part in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
]:
    actual = y_part.to_numpy()
    predicted = tree_model.predict(X_part)

    actual_band = np.digitize(actual, band_cutoffs)
    predicted_band = np.digitize(predicted, band_cutoffs)

    precision, recall, f1, _ = precision_recall_fscore_support(
        actual_band,
        predicted_band,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0,
    )

    results.append({
        "Dataset": split,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(
            np.abs(actual - predicted) <= 0.10 * actual
        ),
        "Band Accuracy (%)": 100 * accuracy_score(actual_band, predicted_band),
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1,
    })

display(pd.DataFrame(results).round(3))

# The final iteration above uses the validation split.
print("Validation results by price band")
print(classification_report(
    actual_band,
    predicted_band,
    labels=[0, 1, 2],
    target_names=band_names,
    zero_division=0,
))

,Dataset,MAE (£),RMSE (£),R²,Within 10% (%),Band Accuracy (%),Macro Precision,Macro Recall,Macro F1
0,Training,2688.961,20153.510,0.635,48.969,89.321,0.895,0.894,0.894
1,Validation,3000.683,40760.608,0.271,47.608,88.826,0.890,0.889,0.889


Validation results by price band
                 precision    recall  f1-score   support

  Under £10,000       0.92      0.88      0.90     23973
£10,000–£24,999       0.87      0.89      0.88     33419
       £25,000+       0.88      0.89      0.89     15167

       accuracy                           0.89     72559
      macro avg       0.89      0.89      0.89     72559
   weighted avg       0.89      0.89      0.89     72559



In [4]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
)
from sklearn.model_selection import GridSearchCV, KFold

param_grid = {
    "model__max_depth": [10, 15, 20],
    "model__min_samples_leaf": [5, 10],
}

tree_search = GridSearchCV(
    estimator=tree_model,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=KFold(n_splits=3, shuffle=True, random_state=42),
    n_jobs=2,
    pre_dispatch=2,
    refit=True,
    verbose=1,
)

tree_search.fit(X_train, y_train)
best_tree = tree_search.best_estimator_

print("Best parameters:", tree_search.best_params_)
print(f"Best CV MAE: £{-tree_search.best_score_:,.2f}")

cv_results = pd.DataFrame(tree_search.cv_results_)
cv_results["CV MAE (£)"] = -cv_results["mean_test_score"]

display(
    cv_results.sort_values("rank_test_score")[
        ["param_model__max_depth", "param_model__min_samples_leaf", "CV MAE (£)"]
    ].round(2)
)

# Compare the original and tuned trees on the same validation rows
actual = y_val.to_numpy()
band_cutoffs = [10_000, 25_000]
actual_band = np.digitize(actual, band_cutoffs)
results = []

for name, model in [("Basic tree", tree_model), ("Tuned tree", best_tree)]:
    predicted = model.predict(X_val)
    predicted_band = np.digitize(predicted, band_cutoffs)

    precision, recall, f1, _ = precision_recall_fscore_support(
        actual_band, predicted_band,
        labels=[0, 1, 2], average="macro", zero_division=0,
    )

    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(
            np.abs(actual - predicted) <= 0.10 * actual
        ),
        "Band Accuracy (%)": 100 * accuracy_score(actual_band, predicted_band),
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1,
    })

display(pd.DataFrame(results).round(3))

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best parameters: {'model__max_depth': 20, 'model__min_samples_leaf': 5}
Best CV MAE: £2,559.38


,param_model__max_depth,param_model__min_samples_leaf,CV MAE (£)
4,20,5,2559.38
5,20,10,2636.84
2,15,5,2946.95
3,15,10,2992.36
1,10,10,3937.77
0,10,5,3967.36


,Model,MAE (£),RMSE (£),R²,Within 10% (%),Band Accuracy (%),Macro Precision,Macro Recall,Macro F1
0,Basic tree,3000.683,40760.608,0.271,47.608,88.826,0.890,0.889,0.889
1,Tuned tree,2485.547,39469.840,0.316,56.349,90.612,0.907,0.907,0.907


In [5]:
results = []
band_cutoffs = [10_000, 25_000]

for model_name, model in [("Basic tree", tree_model), ("Tuned tree", best_tree)]:
    for split_name, features, target in [
        ("Training", X_train, y_train),
        ("Validation", X_val, y_val),
    ]:
        actual = target.to_numpy()
        predicted = model.predict(features)
        absolute_error = np.abs(actual - predicted)

        actual_band = np.digitize(actual, band_cutoffs)
        predicted_band = np.digitize(predicted, band_cutoffs)

        precision, recall, f1, _ = precision_recall_fscore_support(
            actual_band,
            predicted_band,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        )

        results.append({
            "Model": model_name,
            "Split": split_name,
            "MAE (£)": mean_absolute_error(actual, predicted),
            "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
            "R²": r2_score(actual, predicted),
            "Within 10% (%)": 100 * np.mean(absolute_error <= 0.10 * actual),
            "Band Accuracy (%)": 100 * accuracy_score(actual_band, predicted_band),
            "Macro Precision": precision,
            "Macro Recall": recall,
            "Macro F1": f1,
        })

display(pd.DataFrame(results).round(3))

,Model,Split,MAE (£),RMSE (£),R²,Within 10% (%),Band Accuracy (%),Macro Precision,Macro Recall,Macro F1
0,Basic tree,Training,2688.961,20153.510,0.635,48.969,89.321,0.895,0.894,0.894
1,Basic tree,Validation,3000.683,40760.608,0.271,47.608,88.826,0.890,0.889,0.889
2,Tuned tree,Training,1919.373,12563.745,0.858,60.801,91.935,0.920,0.920,0.920
3,Tuned tree,Validation,2485.547,39469.840,0.316,56.349,90.612,0.907,0.907,0.907


In [7]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
)
from sklearn.model_selection import KFold, cross_val_score

min_improvement = 5.0  # Minimum improvement in CV MAE, in pounds

trials = [
    {"model__max_depth": 20},
    {"model__min_samples_leaf": 5},
    {"model__min_samples_split": 30},
    {"model__max_leaf_nodes": 2048},
    {"model__max_features": 0.75},
]

cv = KFold(n_splits=3, shuffle=True, random_state=42)

def cv_mae(model):
    scores = cross_val_score(
        model, X_train, y_train,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=2,
        pre_dispatch=2,
        error_score="raise",
    )
    return -scores.mean()

best_params = {}
best_mae = cv_mae(tree_model)
print(f"Basic tree CV MAE: £{best_mae:,.2f}")

for trial_number, change in enumerate(trials, start=1):
    candidate_params = {**best_params, **change}
    candidate = clone(tree_model).set_params(**candidate_params)

    candidate_mae = cv_mae(candidate)
    improvement = best_mae - candidate_mae

    print(
        f"Trial {trial_number}: {candidate_params} | "
        f"CV MAE £{candidate_mae:,.2f} | "
        f"improvement {improvement:+,.2f} GBP"
    )

    if improvement < min_improvement:
        print("Stopping: improvement is below the cutoff.")
        break

    best_params = candidate_params
    best_mae = candidate_mae

best_tree = tree_model
if best_params:
    best_tree = clone(tree_model).set_params(**best_params)
    best_tree.fit(X_train, y_train)

print("Accepted changes:", best_params)
print(f"Selected CV MAE: £{best_mae:,.2f}")

# Evaluate both trees on the same validation rows
actual = y_val.to_numpy()
cutoffs = [10_000, 25_000]
actual_band = np.digitize(actual, cutoffs)
results = []

for name, model in [("Basic tree", tree_model), ("Selected tree", best_tree)]:
    predicted = model.predict(X_val)
    predicted_band = np.digitize(predicted, cutoffs)

    precision, recall, f1, _ = precision_recall_fscore_support(
        actual_band, predicted_band,
        labels=[0, 1, 2], average="macro", zero_division=0,
    )

    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(
            np.abs(actual - predicted) <= 0.10 * actual
        ),
        "Band Accuracy (%)": 100 * accuracy_score(actual_band, predicted_band),
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1,
    })

display(pd.DataFrame(results).round(3))

Basic tree CV MAE: £2,992.36
Trial 1: {'model__max_depth': 20} | CV MAE £2,636.84 | improvement +355.53 GBP
Trial 2: {'model__max_depth': 20, 'model__min_samples_leaf': 5} | CV MAE £2,559.38 | improvement +77.46 GBP
Trial 3: {'model__max_depth': 20, 'model__min_samples_leaf': 5, 'model__min_samples_split': 30} | CV MAE £2,628.29 | improvement -68.91 GBP
Stopping: improvement is below the cutoff.
Accepted changes: {'model__max_depth': 20, 'model__min_samples_leaf': 5}
Selected CV MAE: £2,559.38


,Model,MAE (£),RMSE (£),R²,Within 10% (%),Band Accuracy (%),Macro Precision,Macro Recall,Macro F1
0,Basic tree,3000.683,40760.608,0.271,47.608,88.826,0.890,0.889,0.889
1,Selected tree,2485.547,39469.840,0.316,56.349,90.612,0.907,0.907,0.907


In [8]:
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error

trials = [
    ("Coarse: increase depth to 26", {"model__max_depth": 26}),
    ("Fine: increase depth to 28", {"model__max_depth": 28}),
    ("Fine: reduce leaf size to 3", {"model__min_samples_leaf": 3}),
]

selected_tree = best_tree
selected_mae = best_mae
accepted_change = False

for label, change in trials:
    candidate = clone(selected_tree).set_params(**change)
    candidate_mae = cv_mae(candidate)
    improvement = selected_mae - candidate_mae

    print(f"{label} | CV MAE £{candidate_mae:,.2f} | Gain £{improvement:+,.2f}")

    if improvement < min_improvement:
        print(f"Stopping: CV MAE improved by less than £{min_improvement:,.2f}.")
        break

    selected_tree = candidate
    selected_mae = candidate_mae
    accepted_change = True

if accepted_change:
    selected_tree.fit(X_train, y_train)

print("\nSelected depth:", selected_tree.get_params()["model__max_depth"])
print("Selected minimum leaf size:", selected_tree.get_params()["model__min_samples_leaf"])
print(f"Selected CV MAE: £{selected_mae:,.2f}")

for name, model in [("Previous winner", best_tree), ("Coarse-to-fine", selected_tree)]:
    val_mae = mean_absolute_error(y_val, model.predict(X_val))
    print(f"{name} validation MAE: £{val_mae:,.2f}")

Coarse: increase depth to 26 | CV MAE £2,432.54 | Gain £+126.84
Fine: increase depth to 28 | CV MAE £2,426.29 | Gain £+6.25
Fine: reduce leaf size to 3 | CV MAE £2,365.59 | Gain £+60.70

Selected depth: 28
Selected minimum leaf size: 3
Selected CV MAE: £2,365.59
Previous winner validation MAE: £2,485.55
Coarse-to-fine validation MAE: £2,267.53


In [9]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    precision_recall_fscore_support,
    r2_score,
)

band_cutoffs = [10_000, 25_000]
band_names = ["Under £10,000", "£10,000 to under £25,000", "£25,000+"]
results = []

for split, features, prices in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
]:
    actual = prices.to_numpy()
    predicted = selected_tree.predict(features)
    absolute_error = np.abs(actual - predicted)

    actual_band = np.digitize(actual, band_cutoffs)
    predicted_band = np.digitize(predicted, band_cutoffs)

    precision, recall, f1, _ = precision_recall_fscore_support(
        actual_band,
        predicted_band,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0,
    )

    results.append({
        "Dataset": split,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "Median Absolute Error (£)": median_absolute_error(actual, predicted),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(absolute_error <= 0.10 * actual),
        "Band Accuracy (%)": 100 * accuracy_score(actual_band, predicted_band),
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1,
    })

    if split == "Validation":
        validation_report = classification_report(
            actual_band,
            predicted_band,
            labels=[0, 1, 2],
            target_names=band_names,
            output_dict=True,
            zero_division=0,
        )

display(pd.DataFrame(results).set_index("Dataset").round(3))

# Validation precision, recall, and F1 for each price band
display(pd.DataFrame(validation_report).T.loc[band_names].round(3))

,MAE (£),RMSE (£),Median Absolute Error (£),R²,Within 10% (%),Band Accuracy (%),Macro Precision,Macro Recall,Macro F1
Dataset,,,,,,,,,
Training,1218.201,9712.451,560.200,0.915,76.893,94.881,0.948,0.95,0.949
Validation,2267.527,39046.992,930.333,0.331,61.681,91.797,0.918,0.92,0.919


,precision,recall,f1-score,support
"Under £10,000",0.937,0.914,0.925,23973.0
"£10,000 to under £25,000",0.909,0.915,0.912,33419.0
"£25,000+",0.909,0.932,0.920,15167.0


In [10]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    precision_recall_fscore_support,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer


def fill_version_text(values):
    return values.fillna("").astype(str)


# Keep the existing preprocessing for every column except version.
preprocessor = selected_tree.named_steps["preprocessor"]
transformers = [
    (
        name,
        clone(transformer),
        [col for col in columns if col != "version"],
    )
    for name, transformer, columns in preprocessor.transformers
]

transformers.append((
    "version_text",
    Pipeline([
        ("fill_missing", FunctionTransformer(fill_version_text)),
        ("vectorizer", CountVectorizer(
            ngram_range=(1, 2),
            min_df=3,
            max_features=15_000,
            token_pattern=r"(?u)\b\w[\w.-]*\b",
        )),
    ]),
    "version",
))

text_preprocessor = clone(preprocessor).set_params(transformers=transformers)

text_tree = clone(selected_tree).set_params(preprocessor=text_preprocessor)
text_tree.fit(X_train, y_train)

cutoffs = [10_000, 25_000]
results = []

for split, features, prices in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
]:
    actual = prices.to_numpy()
    actual_band = np.digitize(actual, cutoffs)

    for name, model in [
        ("One-hot version", selected_tree),
        ("CountVectorizer version", text_tree),
    ]:
        predicted = model.predict(features)
        predicted_band = np.digitize(predicted, cutoffs)
        absolute_error = np.abs(actual - predicted)

        precision, recall, f1, _ = precision_recall_fscore_support(
            actual_band,
            predicted_band,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        )

        results.append({
            "Dataset": split,
            "Version encoding": name,
            "MAE (£)": mean_absolute_error(actual, predicted),
            "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
            "Median Absolute Error (£)": median_absolute_error(actual, predicted),
            "R²": r2_score(actual, predicted),
            "Within 10% (%)": 100 * np.mean(absolute_error <= 0.10 * actual),
            "Band Accuracy (%)": 100 * accuracy_score(actual_band, predicted_band),
            "Macro Precision": precision,
            "Macro Recall": recall,
            "Macro F1": f1,
        })

display(
    pd.DataFrame(results)
    .set_index(["Dataset", "Version encoding"])
    .round(3)
)

MAE (£)   RMSE (£)  \
Dataset    Version encoding                               
Training   One-hot version          1218.201   9712.451   
           CountVectorizer version  1234.665   9721.633   
Validation One-hot version          2267.527  39046.992   
           CountVectorizer version  2278.308  39053.437   

                                    Median Absolute Error (£)     R²  \
Dataset    Version encoding                                            
Training   One-hot version                            560.200  0.915   
           CountVectorizer version                    557.500  0.915   
Validation One-hot version                            930.333  0.331   
           CountVectorizer version                    920.000  0.331   

                                    Within 10% (%)  Band Accuracy (%)  \
Dataset    Version encoding                                             
Training   One-hot version                  76.893             94.881   
           CountVectorizer version          76.785             94.873   
Validation One-hot version                  61.681             91.797   
           CountVectorizer version          62.114             91.899   

                                    Macro Precision  Macro Recall  Macro F1  
Dataset    Version encoding                                                  
Training   One-hot version                    0.948         0.950     0.949  
           CountVectorizer version            0.949         0.949     0.949  
Validation One-hot version                    0.918         0.920     0.919  
           CountVectorizer version            0.920         0.920     0.920

In [11]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

raw = pd.read_csv("uk_used_cars_autotrader.csv")

# Recreate the original holdout using the same file order and seed.
development_raw, holdout_raw = train_test_split(
    raw, test_size=0.2, random_state=42
)

numeric_cols = [
    "mileage_miles", "reg_year", "power_hp", "doors", "seats",
    "previous_owners", "annual_tax_gbp", "co2_gmile",
]
categorical_cols = [
    "make", "model", "version", "fuel", "transmission", "body_type",
    "body_color", "drivetrain", "emission_class", "seller_type",
    "seller_city", "seller_county",
]
feature_cols = numeric_cols + categorical_cols

# These rules operate on each row independently.
# Medians, modes, and category vocabularies are learned later in the pipeline.
def prepare_listings(frame):
    frame = frame.loc[
        frame["condition"].eq("Used") & frame["reg_year"].notna()
    ].copy()

    # Preserve the previous project's owner-count scope.
    frame = frame.loc[
        frame["previous_owners"].isna() | frame["previous_owners"].lt(8)
    ].copy()

    co2 = (frame["co2_gkm"] * 1.609344).round()
    frame["co2_gmile"] = co2.mask(co2.isin([1608, 2092]))

    for col in ["doors", "seats"]:
        frame[col] = frame[col].mask(frame[col].lt(2))

    frame["power_hp"] = frame["power_hp"].mask(frame["power_hp"].le(0))

    for col in categorical_cols:
        frame[col] = (
            frame[col].str.strip()
            .replace({"": "Unknown", "Unlisted": "Unknown"})
            .fillna("Unknown")
        )

    # Only explicitly listed features enter X, excluding price and identifiers.
    return frame[feature_cols], frame["price_gbp"]


X_dev, y_dev = prepare_listings(development_raw)

# Identical predictor rows get the same group, even if their prices differ.
groups = pd.util.hash_pandas_object(X_dev, index=False)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(X_dev, y_dev, groups=groups))

X_train, X_val = X_dev.iloc[train_idx], X_dev.iloc[val_idx]
y_train, y_val = y_dev.iloc[train_idx], y_dev.iloc[val_idx]

assert set(groups.iloc[train_idx]).isdisjoint(groups.iloc[val_idx])

count_cols = ["doors", "seats"]
median_cols = [col for col in numeric_cols if col not in count_cols]

preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), median_cols),
    ("counts", SimpleImputer(strategy="most_frequent"), count_cols),
    ("categories", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

tree = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", DecisionTreeRegressor(
        max_depth=28, min_samples_leaf=3, random_state=42
    )),
])

ridge = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("scale", StandardScaler(with_mean=False)),
        ("model", Ridge(alpha=0.1, solver="lsqr")),
    ]),
    func=np.log1p,
    inverse_func=np.expm1,
)

results = []

for model_name, model in [("Decision Tree", tree), ("Log-price Ridge", ridge)]:
    model.fit(X_train, y_train)

    for split_name, X_part, y_part in [
        ("Training", X_train, y_train),
        ("Validation", X_val, y_val),
    ]:
        predicted = model.predict(X_part)
        actual = y_part.to_numpy()

        results.append({
            "Model": model_name,
            "Split": split_name,
            "MAE (£)": mean_absolute_error(actual, predicted),
            "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
            "R²": r2_score(actual, predicted),
            "Within 10% (%)": 100 * np.mean(
                np.abs(actual - predicted) <= 0.10 * actual
            ),
        })

display(pd.DataFrame(results).round(3))

,Model,Split,MAE (£),RMSE (£),R²,Within 10% (%)
0,Decision Tree,Training,1517.536,15849.517,0.844,70.748
1,Decision Tree,Validation,2362.490,20600.666,0.651,59.182
2,Log-price Ridge,Training,1193.423,6457.892,0.974,77.682
3,Log-price Ridge,Validation,1789.084,17412.042,0.751,71.822


In [12]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Add the feature to copies of your existing splits.
X_train_log = X_train.assign(
    log_mileage=np.log1p(X_train["mileage_miles"])
)
X_val_log = X_val.assign(
    log_mileage=np.log1p(X_val["mileage_miles"])
)

# Clone the baseline so its fitted model stays available for comparison.
ridge_log_mileage = clone(ridge)

# Register log_mileage in the numeric preprocessing branch.
preprocessor = ridge_log_mileage.regressor.named_steps["preprocessor"]
preprocessor.set_params(transformers=[
    (name, transformer, [*columns, "log_mileage"] if name == "numeric" else columns)
    for name, transformer, columns in preprocessor.transformers
])

ridge_log_mileage.fit(X_train_log, y_train)

actual = y_val.to_numpy()
results = []

for name, model, features in [
    ("Baseline log-price Ridge", ridge, X_val),
    ("Log-price Ridge + log mileage", ridge_log_mileage, X_val_log),
]:
    predicted = model.predict(features)
    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(
            np.abs(actual - predicted) <= 0.10 * actual
        ),
    })

comparison = pd.DataFrame(results).set_index("Model")
display(comparison.round(3))

mae_gain = comparison["MAE (£)"].iloc[0] - comparison["MAE (£)"].iloc[1]
print(f"Validation MAE improvement: £{mae_gain:+,.2f}")

,MAE (£),RMSE (£),R²,Within 10% (%)
Model,,,,
Baseline log-price Ridge,1789.084,17412.042,0.751,71.822
Log-price Ridge + log mileage,1784.911,17336.681,0.753,71.856


Validation MAE improvement: £+4.17


In [13]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature = "age_x_log_mileage"

X_train_interaction = X_train_log.assign(
    age_x_log_mileage=(2026 - X_train_log["reg_year"]) * X_train_log["log_mileage"]
)
X_val_interaction = X_val_log.assign(
    age_x_log_mileage=(2026 - X_val_log["reg_year"]) * X_val_log["log_mileage"]
)

# Clone the log-mileage model and add the interaction to its numeric inputs.
ridge_interaction = clone(ridge_log_mileage)
interaction_preprocessor = ridge_interaction.regressor.named_steps["preprocessor"]
interaction_preprocessor.set_params(transformers=[
    (name, transformer, [*columns, feature] if name == "numeric" else columns)
    for name, transformer, columns in interaction_preprocessor.transformers
])

ridge_interaction.fit(X_train_interaction, y_train)

actual = y_val.to_numpy()
results = []

for name, model, features in [
    ("Log-mileage reference", ridge_log_mileage, X_val_log),
    ("Log mileage + age interaction", ridge_interaction, X_val_interaction),
]:
    predicted = model.predict(features)
    absolute_error = np.abs(actual - predicted)

    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(absolute_error <= 0.10 * actual),
    })

comparison = pd.DataFrame(results).set_index("Model")
display(comparison.round(3))

reference, candidate = comparison.iloc[0], comparison.iloc[1]
mae_gain = reference["MAE (£)"] - candidate["MAE (£)"]
within_10_gain = candidate["Within 10% (%)"] - reference["Within 10% (%)"]

print(f"Validation MAE improvement: {mae_gain:+,.2f} GBP")
print(f"Change within 10%: {within_10_gain:+.3f} percentage points")

,MAE (£),RMSE (£),R²,Within 10% (%)
Model,,,,
Log-mileage reference,1784.911,17336.681,0.753,71.856
Log mileage + age interaction,1795.980,18257.744,0.726,72.002


Validation MAE improvement: -11.07 GBP
Change within 10%: +0.147 percentage points


In [14]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train_make_model = X_train_log.assign(
    make_model=X_train_log["make"] + " | " + X_train_log["model"]
)
X_val_make_model = X_val_log.assign(
    make_model=X_val_log["make"] + " | " + X_val_log["model"]
)

# Add the combined category to the existing one-hot encoder.
ridge_make_model = clone(ridge_log_mileage)
make_model_preprocessor = ridge_make_model.regressor.named_steps["preprocessor"]

make_model_preprocessor.set_params(transformers=[
    (name, transformer, [*columns, "make_model"] if name == "categories" else columns)
    for name, transformer, columns in make_model_preprocessor.transformers
])

ridge_make_model.fit(X_train_make_model, y_train)

actual = y_val.to_numpy()
results = []

for name, model, features in [
    ("Log-mileage reference", ridge_log_mileage, X_val_log),
    ("Log mileage + make_model", ridge_make_model, X_val_make_model),
]:
    predicted = model.predict(features)
    absolute_error = np.abs(actual - predicted)

    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(absolute_error <= 0.10 * actual),
    })

comparison = pd.DataFrame(results).set_index("Model")
display(comparison.round(3))

reference, candidate = comparison.iloc[0], comparison.iloc[1]
mae_gain = reference["MAE (£)"] - candidate["MAE (£)"]
within_10_gain = candidate["Within 10% (%)"] - reference["Within 10% (%)"]

print(f"Validation MAE improvement: {mae_gain:+,.2f} GBP")
print(f"Change within 10%: {within_10_gain:+.3f} percentage points")

,MAE (£),RMSE (£),R²,Within 10% (%)
Model,,,,
Log-mileage reference,1784.911,17336.681,0.753,71.856
Log mileage + make_model,1768.183,17158.022,0.758,71.865


Validation MAE improvement: +16.73 GBP
Change within 10%: +0.009 percentage points


In [15]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error

feature = "battery_capacity_kwh"

def add_battery_capacity(frame):
    fuel = frame["fuel"].fillna("").str.strip().str.lower()
    electric = fuel.eq("electric")
    plug_in = fuel.str.contains(r"plug[\s-]?in", regex=True)

    # Extract values such as 66.5kWh or 82 kWh from the version text.
    capacity = frame["version"].str.extract(
        r"(?i)\b(\d+(?:\.\d+)?)\s*kwh\b",
        expand=False,
    )
    capacity = pd.to_numeric(capacity, errors="coerce")
    capacity = capacity.where(electric | plug_in)

    # Zero means no capacity was extracted for this experiment.
    return frame.assign(battery_capacity_kwh=capacity.fillna(0.0))


X_train_battery = add_battery_capacity(X_train_make_model)
X_val_battery = add_battery_capacity(X_val_make_model)

ridge_battery = clone(ridge_make_model)
battery_preprocessor = ridge_battery.regressor.named_steps["preprocessor"]

battery_preprocessor.set_params(transformers=[
    (name, transformer, [*columns, feature] if name == "numeric" else columns)
    for name, transformer, columns in battery_preprocessor.transformers
])

ridge_battery.fit(X_train_battery, y_train)

actual = y_val.to_numpy()
baseline_pred = ridge_make_model.predict(X_val_make_model)
battery_pred = ridge_battery.predict(X_val_battery)

fuel = X_val_battery["fuel"].fillna("").str.strip().str.lower()
has_capacity = X_val_battery[feature].gt(0).to_numpy()

groups = [
    ("All listings", np.ones(len(actual), dtype=bool)),
    ("Electric", fuel.eq("electric").to_numpy()),
    ("Plug-in hybrid", fuel.str.contains(r"plug[\s-]?in", regex=True).to_numpy()),
]

results = []

for group, mask in groups:
    if not mask.any():
        continue

    for model, predicted in [
        ("Log mileage + make_model", baseline_pred),
        ("+ Battery capacity", battery_pred),
    ]:
        error = np.abs(actual[mask] - predicted[mask])

        results.append({
            "Group": group,
            "Model": model,
            "Listings": int(mask.sum()),
            "Capacity extracted (%)": 100 * has_capacity[mask].mean(),
            "MAE (£)": mean_absolute_error(actual[mask], predicted[mask]),
            "RMSE (£)": np.sqrt(mean_squared_error(actual[mask], predicted[mask])),
            "Within 10% (%)": 100 * np.mean(error <= 0.10 * actual[mask]),
        })

display(pd.DataFrame(results).set_index(["Group", "Model"]).round(3))

Listings  Capacity extracted (%)  \
Group          Model                                                        
All listings   Log mileage + make_model     73695                  10.390   
               + Battery capacity           73695                  10.390   
Electric       Log mileage + make_model      4429                  89.727   
               + Battery capacity            4429                  89.727   
Plug-in hybrid Log mileage + make_model      3719                  99.032   
               + Battery capacity            3719                  99.032   

                                          MAE (£)   RMSE (£)  Within 10% (%)  
Group          Model                                                          
All listings   Log mileage + make_model  1768.183  17158.022          71.865  
               + Battery capacity        1768.409  17161.501          71.869  
Electric       Log mileage + make_model  1653.247   4571.150          86.634  
               + Battery capacity        1652.633   4571.382          86.679  
Plug-in hybrid Log mileage + make_model  2101.219   7615.880          86.045  
               + Battery capacity        2101.225   7617.202          86.045

In [16]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Start with these checks. Add more columns to the list if needed.
ablation_tests = [
    ["seller_city"],
    ["seller_county"],
    ["body_color"],
    ["emission_class"],
    ["annual_tax_gbp"],
    ["version"],
    ["seller_city", "seller_county"],
]

actual = y_val.to_numpy()
price_cutoffs = [10_000, 25_000]
actual_bands = np.digitize(actual, price_cutoffs)


def validation_metrics(predicted):
    absolute_error = np.abs(actual - predicted)

    return {
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
        "Within 10% (%)": 100 * np.mean(absolute_error <= 0.10 * actual),
        "Band Accuracy (%)": 100 * np.mean(
            np.digitize(predicted, price_cutoffs) == actual_bands
        ),
    }


baseline = validation_metrics(ridge_make_model.predict(X_val_make_model))
results = [{"Removed": "None — baseline", **baseline, "MAE gain (£)": 0.0}]

for columns_to_remove in ablation_tests:
    candidate = clone(ridge_make_model)
    preprocessor = candidate.regressor.named_steps["preprocessor"]

    # Remove these features from the columns used by the pipeline.
    preprocessor.set_params(transformers=[
        (
            name,
            transformer,
            [column for column in columns if column not in columns_to_remove],
        )
        for name, transformer, columns in preprocessor.transformers
    ])

    candidate.fit(X_train_make_model, y_train)
    scores = validation_metrics(candidate.predict(X_val_make_model))

    removed = " + ".join(columns_to_remove)
    mae_gain = baseline["MAE (£)"] - scores["MAE (£)"]

    results.append({"Removed": removed, **scores, "MAE gain (£)": mae_gain})
    print(f"Removed {removed}: MAE gain £{mae_gain:+,.2f}", flush=True)

ablation_results = pd.DataFrame(results).sort_values(
    "MAE gain (£)", ascending=False
)
display(ablation_results.round(3))

Removed seller_city: MAE gain £-20.98
Removed seller_county: MAE gain £-1.80
Removed body_color: MAE gain £-2.63
Removed emission_class: MAE gain £+3.40
Removed annual_tax_gbp: MAE gain £+0.23
Removed version: MAE gain £-931.02
Removed seller_city + seller_county: MAE gain £-20.38


,Removed,MAE (£),RMSE (£),R²,Within 10% (%),Band Accuracy (%),MAE gain (£)
4,emission_class,1764.782,17201.323,0.757,71.864,94.133,3.402
5,annual_tax_gbp,1767.951,17187.273,0.757,71.797,94.134,0.232
0,None — baseline,1768.183,17158.022,0.758,71.865,94.115,0.000
2,seller_county,1769.988,17169.429,0.758,71.814,94.154,-1.805
3,body_color,1770.816,17215.415,0.756,71.869,94.127,-2.633
7,seller_city + seller_county,1788.567,17929.903,0.736,71.511,94.059,-20.384
1,seller_city,1789.164,17865.134,0.738,71.615,94.134,-20.980
6,version,2699.208,18054.805,0.732,53.043,90.941,-931.024


In [17]:
import joblib
import sklearn
from sklearn.base import clone

# Recreate the selected configuration from the ablation experiment.
final_model = clone(ridge_make_model)
preprocessor = final_model.regressor.named_steps["preprocessor"]

preprocessor.set_params(transformers=[
    (name, transformer, [col for col in columns if col != "emission_class"])
    for name, transformer, columns in preprocessor.transformers
])

final_model.fit(X_train_make_model, y_train)

model_path = "used_car_log_price_ridge.joblib"
joblib.dump(
    {"model": final_model, "sklearn_version": sklearn.__version__},
    model_path,
    compress=3,
)

print(f"Saved {model_path}")

Saved used_car_log_price_ridge.joblib


In [18]:
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# Load the saved model and the original 36-column test CSV.
saved = joblib.load("used_car_log_price_ridge.joblib")
model = saved["model"]
test_raw = pd.read_csv("test.csv")

# Apply the same row filters and cleaning rules used in training.
X_eligible, y_eligible = prepare_listings(test_raw)

if X_eligible.empty:
    raise ValueError("No test listings passed the training eligibility rules.")

# Sample eligible rows and keep their asking prices in the same order.
X_sample = X_eligible.sample(n=min(1_000, len(X_eligible)), random_state=42)
y_sample = y_eligible.loc[X_sample.index]

# Recreate the features used by the saved model.
X_sample = X_sample.assign(
    log_mileage=np.log1p(X_sample["mileage_miles"]),
    make_model=X_sample["make"] + " | " + X_sample["model"],
)

actual = y_sample.to_numpy()
predicted = model.predict(X_sample)
absolute_error = np.abs(actual - predicted)

# Use the same price bands as validation.
cutoffs = [10_000, 25_000]
actual_band = np.digitize(actual, cutoffs)
predicted_band = np.digitize(predicted, cutoffs)

print(f"Raw test listings: {len(test_raw):,}")
print(f"Eligible listings: {len(X_eligible):,}")
print(f"Sample evaluated: {len(X_sample):,}")
print(f"\nMAE: £{mean_absolute_error(actual, predicted):,.2f}")
print(f"RMSE: £{np.sqrt(mean_squared_error(actual, predicted)):,.2f}")
print(f"R²: {r2_score(actual, predicted):.3f}")
print(f"Within 10% of asking price: {np.mean(absolute_error <= 0.10 * actual):.2%}")
print(f"Price-band accuracy: {accuracy_score(actual_band, predicted_band):.2%}")

print("\nPrecision, recall, and F1 by price band")
print(classification_report(
    actual_band,
    predicted_band,
    labels=[0, 1, 2],
    target_names=["Under £10,000", "£10,000 to under £25,000", "£25,000+"],
    digits=3,
    zero_division=0,
))

Raw test listings: 95,995
Eligible listings: 92,098
Sample evaluated: 1,000

MAE: £1,520.06
RMSE: £5,478.03
R²: 0.877
Within 10% of asking price: 70.90%
Price-band accuracy: 93.60%

Precision, recall, and F1 by price band
                          precision    recall  f1-score   support

           Under £10,000      0.967     0.965     0.966       339
£10,000 to under £25,000      0.936     0.928     0.932       470
                £25,000+      0.883     0.906     0.894       191

                accuracy                          0.936      1000
               macro avg      0.929     0.933     0.931      1000
            weighted avg      0.936     0.936     0.936      1000



In [19]:
residual = actual - predicted

# Percentage residual requires a positive asking price.
positive_price = actual > 0
residual_pct = 100 * residual[positive_price] / actual[positive_price]

print(f"Average residual: {residual.mean():+,.2f} GBP")
print(f"Average percentage residual: {residual_pct.mean():+.2f}%")

Average residual: -208.43 GBP
Average percentage residual: -3.98%


In [20]:
comparison = pd.DataFrame({
    "Actual price (£)": actual,
    "Predicted price (£)": predicted,
})

display(comparison.head(15).round(2))

,Actual price (£),Predicted price (£)
0,8500,7843.63
1,2100,2550.72
2,6495,4491.65
3,15295,15387.32
4,2480,3777.88
5,6995,7537.96
6,13850,17788.44
7,18108,17404.19
8,19495,20070.39
9,3795,3647.22


In [21]:
comparison = pd.DataFrame({
    "Actual price (£)": actual,
    "Predicted price (£)": predicted,
    "Residual (£)": actual - predicted,
})

display(comparison.head(15).round(2))

,Actual price (£),Predicted price (£),Residual (£)
0,8500,7843.63,656.37
1,2100,2550.72,-450.72
2,6495,4491.65,2003.35
3,15295,15387.32,-92.32
4,2480,3777.88,-1297.88
5,6995,7537.96,-542.96
6,13850,17788.44,-3938.44
7,18108,17404.19,703.81
8,19495,20070.39,-575.39
9,3795,3647.22,147.78
